# **Lab 2: Efficient Image Classification Using ExecuTorch on Raspberry Pi**
### CPU Inference on Raspberry Pi 5 (Arm-powered) - MobileNetV2 CNN, Quantization, and Example Application

## **Introduction**

Welcome to the **Efficient Image Classification Using ExecuTorch on Raspberry Pi** lab. In this hands-on session, developed by Professor Marcelo Rovai (UNIFEI & Edge AI Foundation), you will explore a typical deployment route for image classification on a Raspberry Pi.

Image classification is a fundamental computer vision task that powers countless real-world applications, from quality control in manufacturing to wildlife monitoring, medical diagnostics, and smart home devices. In the Edge AI landscape, the ability to run these models efficiently on resource-constrained devices has become increasingly critical for privacy-preserving, low-latency applications. This lab will utilize MobileNetV2, a convolutional neural network model. If you are unfamiliar with the different types of neural networks, Arm's free Intro to AI course on edX is a good starting point: [Intro to AI](https://www.edx.org/learn/computer-science/arm-education-introduction-to-ai).

This lab follows-on from Lab 1, which showcased how utilizing XNNPACK, which in turn leverages Kleidi AI, can provide an ExecuTorch deployment with similar or superior performance to PyTorch. More importantly, it allows for PyTorch models to be deployed on resource-constrained Edge and embedded devices that may not have the required memory to support full PyTorch and its dependencies.

This lab will recap the XNNPACK workflow for MobileNetV2, and introduce quantization, a staple technique used for Edge deployment, which would be expected in typical ExecuTorch applications. The lab will also discuss Tensorflow Lite and how it compares with ExecuTorch.

**Requirements**: To complete this lab, you are recommended to utilize a Raspberry Pi 4 or 5, running Raspberry Pi OS with a Raspberry Pi Camera. If you are only running the initial examples and not using the PiCamera, a MacBook Pro M4 or similar Arm-based Mac should work.

### **Lab Objectives:** 

The objectives of this lab are as follows:

1. **Refresh on ExecuTorch deployment workflow and observe the behavior when lowering a CNN**:
   - Reinforce prior learning on the typical workflow when using ExecuTorch on Arm-based CPUs with XNNPACK.
   - Understand the clean delegation of MobileNetV2 to XNNPACK, and the implication for performance.
   - Identify the similarities and differences between ExecuTorch and TensorFlow Lite.

2. **Understand how and why quantization can be used with ExecuTorch for compute and memory efficiency**:
   - Understand quantization, how it is approached for ExecuTorch, and where that differs from quantization in PyTorch.
   - Observe and understand the performance gains and memory savings.

3. **Identify next steps for applying your understanding and learning more**:
   - Using both pre-provided images and the Raspberry Pi Camera, perform basic, efficient image classifications. With these and the prior transformer-based examples, you should be in a position to attempt the creation of simple demo applications across natural language and vision.
   - Be equipped to move on to the next lab, which will introduce a non-CPU backend, the Ethos-U NPU

### **Prerequisites**

To follow this lab, you should have a basic understanding of:

- **Python programming**: Experience writing and running Python scripts.
- **AI/ML principles**: Basic understanding of AI/ML and model types.

### **Recommended**
- It is recommended to complete Lab 1 prior to this lab.
- Non-essential but recommended prior to this lab is to complete the [Optimizing GenAI on Arm](https://www.edx.org/learn/computer-science/arm-education-ai-on-arm) course on edX or Coursera.

> Note: It is assumed that the same Jupyter Kernel is used throughout this lab and that all cells are run sequentially

### **Framework Comparison: ExecuTorch vs TensorFlow Lite**

Before diving into the lab, we will briefly consider TensorFlow Lite, another framework and runtime which you may be familiar with. For more detail on Image Classification with Tensorflow Lite, you can explore more of Professor Rovai's work: [Image Classification Fundamentals](https://mjrovai.github.io/EdgeML_Made_Ease_ebook/raspi/image_classification/image_classification_fund.html).

Understanding when to choose each framework is crucial for effective edge deployment:

| Feature                  | ExecuTorch                             | TensorFlow Lite                 |
| ------------------------ | -------------------------------------- | ------------------------------- |
| **Training Framework**   | PyTorch                                | TensorFlow/Keras                |
| **Maturity**             | Newer (2023+)                          | Mature (2017+)                  |
| **Model Format**         | `.pte`                                 | `.tflite (.lite)`               |
| **Quantization**         | PyTorch native quantization            | TF quantization-aware training  |
| **Community**            | Rapidly growing                        | Large, established              |
| **Hardware Support**     | Expanding quickly                      | Extensive, mature               |
| **Learning Curve**       | Easier for PyTorch users               | Easier for TF/Keras users       |
| **Documentation**        | Growing, modern                        | Comprehensive, mature           |
| **Industry Adoption**    | Increasing in research                 | Widespread in production        |

**The Reality: Both Are Excellent Choices**

In practice, both frameworks achieve similar goals with different philosophies. Our choice often comes down to:

1. Our training framework preference.
2. Team expertize and existing infrastructure.
3. Specific hardware requirements.
4. Project timeline and maturity needs.

To keep the notebook output clean and focus on learning, we will suppress some verbose, non-critical warnings.

In [ ]:
import warnings
import logging
import os

# Suppress Python warnings (including FutureWarning)
warnings.filterwarnings("ignore")

# Reduce logging verbosity
logging.getLogger().setLevel(logging.ERROR)

# Suppress Python warning environment noise
os.environ["PYTHONWARNINGS"] = "ignore"

## **MobileNetV2 Efficient Image Classification Example**

Install required libraries and create your working directories with the following script:

In [ ]:
%%bash

# Install various libraries.
pip install torch torchvision executorch transformers psutil matplotlib torchao

# Create working directories.
mkdir -p Image_Lab_ExecuTorch
cd Image_Lab_ExecuTorch
mkdir -p img_class
cd img_class
mkdir -p mobilenet
cd mobilenet
mkdir -p models images


Load an image from the internet, for example, a cat: `"https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"`

And save it in the images folder as "cat.jpg":

In [ ]:
%%bash
cd Image_Lab_ExecuTorch/img_class/mobilenet
wget "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg" \
     -O ./images/cat.jpg

As in Lab 1, we'll compare a PyTorch baseline against ExecuTorch with the XNNPACK delegate. The PyTorch steps are as follow:

1. **First run** - Downloads the model, labels, and saves them.
2. **Preprocessing** - MobileNetV2 expects 224x224 images with ImageNet normalization. 
3. **torch.no_grad()** - Disables gradient calculation for faster inference.
4. **Timing** - Measures only inference time, not preprocessing.
5. **Softmax** - Converts raw outputs to probabilities. 
6. **Top-5** - Shows the five most likely classes.

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from PIL import Image
import time
import json
import urllib.request
import os
import numpy as np
import matplotlib.pyplot as plt

# Paths.
MODEL_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/mobilenet_v2.pth"
LABELS_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/imagenet_labels.json"
IMAGE_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/images/cat.jpg"

# Benchmark settings.
NUM_RUNS = 50
WARMUP_RUNS = 10

# Download and save ImageNet labels.
if not os.path.exists(LABELS_PATH):
    print("Downloading ImageNet labels...")
    LABELS_URL = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
    with urllib.request.urlopen(LABELS_URL) as url:
        labels = json.load(url)

    # Save labels locally.
    with open(LABELS_PATH, 'w') as f:
        json.dump(labels, f)
    print(f"Labels saved to {LABELS_PATH}")
else:
    print("Loading labels from disk...")
    with open(LABELS_PATH, 'r') as f:
        labels = json.load(f)

# Load or download model.
if not os.path.exists(MODEL_PATH):
    print("Downloading MobileNetV2 model...")
    weights = MobileNet_V2_Weights.IMAGENET1K_V1
    model = mobilenet_v2(weights=weights)
    model.eval()
    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Model saved to {MODEL_PATH}")
else:
    print("Loading model from disk...")
    model = models.mobilenet_v2()
    model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
    model.eval()

# Define image preprocessing.
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load and preprocess image.
print(f"\nLoading image from {IMAGE_PATH}...")
img = Image.open(IMAGE_PATH)
img_tensor = preprocess(img)
batch = img_tensor.unsqueeze(0)

# Perform inference with timing.
print("Running inference...")
start_time = time.time()

with torch.no_grad():
    output = model(batch)

inference_time = (time.time() - start_time) * 1000

# Get predictions.
probabilities = torch.nn.functional.softmax(output[0], dim=0)
top5_prob, top5_idx = torch.topk(probabilities, 5)

Display the results of the initial Python benchmark below:

In [ ]:
# Display results.
print("\n" + "=" * 50)
print("CLASSIFICATION RESULTS")
print("=" * 50)
print("Top 5 Predictions:")
print("-" * 50)

for i in range(5):
    idx = top5_idx[i].item()
    prob = top5_prob[i].item()
    print(f"{i+1}. {labels[idx]:20s} - {prob*100:.2f}%")

print("=" * 50)

# -------------------------
# Benchmark + graphs.
# -------------------------
print(f"\nBenchmarking inference time distribution...")
print(f"Warmup runs: {WARMUP_RUNS}")
print(f"Number of runs: {NUM_RUNS}\n")

# Warmup.
print("Warming up...")
for _ in range(WARMUP_RUNS):
    with torch.no_grad():
        _ = model(batch)

# Benchmark.
print("Running benchmark...")
times = []
for i in range(NUM_RUNS):
    start = time.time()
    with torch.no_grad():
        _ = model(batch)
    times.append(time.time() - start)

times = np.array(times) * 1000  # Convert to ms.

# Print statistics.
print("\n" + "=" * 50)
print("BENCHMARK RESULTS")
print("=" * 50)
print(f"  Mean:   {times.mean():.2f} ms")
print(f"  Median: {np.median(times):.2f} ms")
print(f"  Std:    {times.std():.2f} ms")
print(f"  Min:    {times.min():.2f} ms")
print(f"  Max:    {times.max():.2f} ms")
print("=" * 50)

# Plot distribution.
plt.figure(figsize=(12, 4))

# Histogram.
plt.subplot(1, 2, 1)
plt.hist(times, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(times.mean(), color='red', linestyle='--',
            label=f'Mean: {times.mean():.2f} ms')
plt.xlabel('Inference Time (ms)')
plt.ylabel('Frequency')
plt.title('Inference Time Distribution')
plt.legend()
plt.grid(alpha=0.3)

# Time series.
plt.subplot(1, 2, 2)
plt.plot(times, marker='o', markersize=3, alpha=0.6)
plt.axhline(times.mean(), color='red', linestyle='--',
            label=f'Mean: {times.mean():.2f} ms')
plt.xlabel('Run Number')
plt.ylabel('Inference Time (ms)')
plt.title('Inference Time Over Runs')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

As in Lab 1, we export the PyTorch model to the `.pte` ExecuTorch format ourselves, with full control over the export process. This differs from TensorFlow Lite, where we typically utilize pre-converted .tflite models.

The ExecuTorch export process involves several steps and is recapped below:

1. `export()` → Captures the model graph.
2. `to_edge()` → Converts to Edge dialect.
3. `to_ExecuTorch()` → Lowers to the ExecuTorch format.
4. `.buffer` → Retrieves the binary data to save.

```
PyTorch Model (.pt/.pth)
          ↓
    torch.export()         # Export to ExportedProgram
          ↓
    to_edge()              # Convert to Edge dialect
          ↓
    to_ExecuTorch()        # Generate ExecuTorch program
          ↓
   .pte file               # Ready for edge deployment
```

In the first lab, we observed that by default ExecuTorch doesn't provide a speed-up compared with PyTorch. That export path produces a generic ExecuTorch CPU graph with reference kernels and no backend optimizations or fusions, so significantly higher latency than PyTorch is expected for MobileNetV2 on a Pi 5.

ExecuTorch is designed to excel when delegated to a backend, such as  XNNPACK, where large subgraphs are lowered into highly optimized kernels.

Let's export the PyTorch model with XNNPACK, a CPU‑optimized backend, and run with that backend enabled:

In [ ]:
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

# Paths.
PYTORCH_MODEL_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/mobilenet_v2.pth"
EXECUTORCH_MODEL_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/mobilenet_v2_xnnpack.pte"

print("Loading PyTorch model...")
if not os.path.exists(PYTORCH_MODEL_PATH):
    raise FileNotFoundError(
        f"PyTorch model weights not found at {PYTORCH_MODEL_PATH}. "
        "Make sure you've saved mobilenet_v2.pth there first."
    )

# Build model skeleton + load weights.
model = models.mobilenet_v2(weights=None)
state = torch.load(PYTORCH_MODEL_PATH, map_location="cpu")
model.load_state_dict(state)
model.eval()

# Create example input.
example_input = (torch.randn(1, 3, 224, 224),)

print("Exporting to ExecuTorch with XNNPACK backend...")

# Step 1: Export with torch.export.
print("  1. Capturing model with torch.export...")
exported_program = export(model, example_input)

# Step 2: Transform, partition, and lower to Edge (XNNPACK delegation configured here).
print("  2. Transforming with XNNPACK partitioner...")
edge_manager = to_edge_transform_and_lower(
    exported_program,
    partitioner={"forward": [XnnpackPartitioner()]},
)

# Step 3: Lower Edge -> ExecuTorch program.
print("  3. Lowering to ExecuTorch...")
executorch_program = edge_manager.to_executorch()

# Step 4: Save .pte.
print("  4. Saving to .pte file...")
os.makedirs(os.path.dirname(EXECUTORCH_MODEL_PATH), exist_ok=True)

# Delegation inspection.
gm = edge_manager.exported_program().graph_module

delegate_op = torch.ops.higher_order.executorch_call_delegate

delegate_calls = 0
fallback_ops = 0

for n in gm.graph.nodes:
    if n.op == "call_function":
        if n.target == delegate_op:
            delegate_calls += 1
        else:
            fallback_ops += 1

print(f"\nXNNPACK delegate call sites : {delegate_calls}")
print(f"Fallback call_function ops  : {fallback_ops}\n\n")

buf = getattr(executorch_program, "buffer", None)
if buf is None:
    buf = getattr(executorch_program, "_buffer", None)
if buf is None:
    raise AttributeError(
        "Could not find the serialized program buffer on the lowered ExecuTorch program. "
        f"Available attrs: {dir(executorch_program)}"
    )

with open(EXECUTORCH_MODEL_PATH, "wb") as f:
    f.write(buf)

print(f"\nModel successfully exported to {EXECUTORCH_MODEL_PATH}")

# Display file size.
pytorch_size = os.path.getsize(PYTORCH_MODEL_PATH) / (1024 * 1024)
executorch_size = os.path.getsize(EXECUTORCH_MODEL_PATH) / (1024 * 1024)

print("\n" + "=" * 50)
print("MODEL SIZE")
print("=" * 50)
print(f"ExecuTorch+XNNPACK:      {executorch_size:.2f} MB")
print("=" * 50)

The MobileNetV2 delegation to XNNPACK results demonstrate a much cleaner and more consolidated execution pattern compared with the transformer-based models in Lab 1. You will likely see small variations in your delegation compared to ours, due to differences in library versions, build configuration, or environment setup. In our run, MobileNetV2 has two XNNPACK delegate call sites and five default ExecuTorch `call_function` operations.

The vast majority of the model’s computation has been captured inside large, contiguous delegated regions. This indicates that most of MobileNetV2’s heavy compute — primarily convolutions, depthwise convolutions, and pointwise (1×1) convolutions — maps cleanly onto XNNPACK’s optimized CPU kernels. As a result, there are very few transitions between the delegated backend and the default runtime, which minimizes boundary overhead and typically leads to more predictable performance improvements.

In [ ]:
from executorch.extension.pybindings.portable_lib import _load_for_executorch

def run_executorch_model(EXECUTORCH_MODEL_PATH, IMAGE_PATH):
    
    # Paths.
    LABELS_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/imagenet_labels.json"

    # Benchmark settings.
    NUM_RUNS = 50
    WARMUP_RUNS = 10

    # Load labels.
    print("Loading labels...")
    with open(LABELS_PATH, 'r') as f:
        labels = json.load(f)

    # Load ExecuTorch model.
    print(f"Loading ExecuTorch model from {EXECUTORCH_MODEL_PATH}...")
    model = _load_for_executorch(EXECUTORCH_MODEL_PATH)

    # Preprocess
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    # Load and preprocess image.
    print(f"Loading image from {IMAGE_PATH}...")
    img = Image.open(IMAGE_PATH).convert("RGB")

    img_tensor = preprocess(img)
    batch = img_tensor.unsqueeze(0)

    # -------------------------
    # Single inference + top-5.
    # -------------------------
    print("Running ExecuTorch inference...")
    start_time = time.time()

    # ExecuTorch expects a tuple of inputs.
    with torch.no_grad():
        output = model.forward((batch,))

    inference_time = (time.time() - start_time) * 1000  # Convert to ms.

    # Get predictions.
    output_tensor = output[0]  # ExecuTorch returns a list.
    probabilities = torch.nn.functional.softmax(output_tensor[0], dim=0)
    top5_prob, top5_idx = torch.topk(probabilities, 5)

    # Display results.
    print("\n" + "=" * 50)
    print("CLASSIFICATION RESULTS")
    print("=" * 50)
    print("Top 5 Predictions:")
    print("-" * 50)

    for i in range(5):
        idx = top5_idx[i].item()
        prob = top5_prob[i].item()
        print(f"{i+1}. {labels[idx]:20s} - {prob*100:.2f}%")

    print("=" * 50)

    # -------------------------
    # Benchmark + graphs.
    # -------------------------
    print(f"\nBenchmarking inference time distribution...")
    print(f"Warmup runs: {WARMUP_RUNS}")
    print(f"Number of runs: {NUM_RUNS}\n")

    # Warmup.
    print("Warming up...")
    for _ in range(WARMUP_RUNS):
        with torch.no_grad():
            _ = model.forward((batch,))

    # Benchmark.
    print("Running benchmark...")
    times = []
    for i in range(NUM_RUNS):
        start = time.time()
        with torch.no_grad():
            _ = model.forward((batch,))
        times.append(time.time() - start)

    times = np.array(times) * 1000  # Convert to ms.

    # Print statistics.
    print("\n" + "=" * 50)
    print("BENCHMARK RESULTS")
    print("=" * 50)
    print(f"  Mean:   {times.mean():.2f} ms")
    print(f"  Median: {np.median(times):.2f} ms")
    print(f"  Std:    {times.std():.2f} ms")
    print(f"  Min:    {times.min():.2f} ms")
    print(f"  Max:    {times.max():.2f} ms")
    print("=" * 50)

    # Plot distribution.
    plt.figure(figsize=(12, 4))

    # Histogram.
    plt.subplot(1, 2, 1)
    plt.hist(times, bins=20, edgecolor='black', alpha=0.7)
    plt.axvline(times.mean(), color='red', linestyle='--',
                label=f'Mean: {times.mean():.2f} ms')
    plt.xlabel('Inference Time (ms)')
    plt.ylabel('Frequency')
    plt.title('Inference Time Distribution')
    plt.legend()
    plt.grid(alpha=0.3)

    # Time series.
    plt.subplot(1, 2, 2)
    plt.plot(times, marker='o', markersize=3, alpha=0.6)
    plt.axhline(times.mean(), color='red', linestyle='--',
                label=f'Mean: {times.mean():.2f} ms')
    plt.xlabel('Run Number')
    plt.ylabel('Inference Time (ms)')
    plt.title('Inference Time Over Runs')
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

EXECUTORCH_MODEL_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/mobilenet_v2_xnnpack.pte"
IMAGE_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/images/cat.jpg"
run_executorch_model(EXECUTORCH_MODEL_PATH, IMAGE_PATH)

Now the ExecuTorch runtime detects the backend automatically from the .pte file metadata, and we have achieved faster inference, as in Lab 1. This latency is, in fact, several times faster than PyTorch. In this case, we also obtained greater performance with ExecuTorch and XNNPACK on MacBook, not just on a Raspberry Pi. This is because, as shown previously, MobileNetV2 delegates extremely well to XNNPACK, so even on a higher-specification device it outperforms PyTorch.

We can now add quantization to get an even smaller model size while maintaining, or even increasing, this speed.

### Model Quantization

Quantization reduces model size and **can** further improve inference speed. ExecuTorch supports PyTorch's native quantization.

**Quantization Overview**

Quantization is a technique that reduces the precision of numbers used in a model’s computations and stored weights, typically from 32-bit floats to 8-bit integers. This reduces the model’s memory footprint, speeds up inference, and lowers power consumption, often with minimal loss in accuracy.

Quantization is especially important for deploying models on edge devices such as wearables, embedded systems, and microcontrollers, which often have limited compute, memory, and battery capacity. By quantizing models, we can make them significantly more efficient and better suited to these resource-constrained environments. For more information on advanced quantisation techniques, check out Arm's PyTorch [Advanced Quantization resource](https://github.com/arm-university/Advanced-AI-Hardware-Software-Co-Design).

**Quantization in ExecuTorch**

ExecuTorch uses [torchao](https://github.com/pytorch/ao/tree/main/torchao) as its quantization library. This integration allows ExecuTorch to leverage PyTorch-native tools for preparing, calibrating, and converting quantized models.

Quantization in ExecuTorch is backend-specific. Each backend defines how models should be quantized based on its hardware capabilities. Most ExecuTorch backends use the torchao PT2E [quantization](https://docs.pytorch.org/executorch/1.0/quantization-overview.html) flow, which works with models exported with torch.export and enables tailored quantization for each backend. 

For a quantized XNNPACK `.pte`, we need a different pipeline: PT2E quantization (with `XNNPACKQuantizer`), then lowering with `XnnpackPartitioner` before `to_executorch()`. Otherwise, we will hit errors or get an undelegated model.

For the conversion, we need to: (1) calibrate with real, preprocessed images, and (2) compute the quantized `.pte` size after you writing the file. 

Calibration is the step where observers collect activation statistics, for example min/max values, by running representative data (in this case, images) through the prepared model.

These statistics are used to compute quantization parameters (scale and zero-point) for each tensor.

After calibration, convert_pt2e() replaces floating-point operations with quantized operations using the computed parameters, producing a statically quantized model ready for ExecuTorch lowering and deployment.

First, let us create a small `calib_images/` folder (e.g., 50–100 natural images across a few classes). A simple way is to reuse an existing dataset, such as CIFAR‑10, and save 50–100 images into `calib_images/` with an ImageNet‑style folder layout.


In [ ]:
from pathlib import Path

from torchvision import datasets, transforms
from torchvision.utils import save_image

# Where to store calibration images.
OUT_ROOT = Path("Image_Lab_ExecuTorch/calib_images")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# 1) Load a small, natural-image dataset (CIFAR-10).
transform = transforms.ToTensor() 
dataset = datasets.CIFAR10(
    root="Image_Lab_ExecuTorch/data",
    train=True,
    download=True,
    transform=transform,
)

# 2) Map label index -> class name (CIFAR-10 has 10 classes).
classes = dataset.classes  # ['airplane', 'automobile', ..., 'truck'].

# 3) Choose how many classes and images per class.
num_classes = 10
images_per_class = 10   # 10 x 10 = 100 images.

# 4) Collect and save images.
counts = {cls: 0 for cls in classes[:num_classes]}

for img, label in dataset:
    cls_name = classes[label]
    if cls_name not in counts:
        continue
    if counts[cls_name] >= images_per_class:
        continue

    # Make class subdir.
    class_dir = OUT_ROOT / cls_name
    class_dir.mkdir(parents=True, exist_ok=True)

    idx = counts[cls_name]
    out_path = class_dir / f"img_{idx:04d}.jpg"
    save_image(img, out_path)

    counts[cls_name] += 1

    if all(counts[c] >= images_per_class for c in counts):
        break

print("Saved calibration images:")
for cls_name, n in counts.items():
    print(f"  {cls_name}: {n} images")
print(f"\nRoot folder: {OUT_ROOT.resolve()}")

Let's quantize to INT8 and run the inference:

In [ ]:
import os
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import random
from torchao.quantization.pt2e import move_exported_model_to_eval

from torch.export import export
from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    get_symmetric_quantization_config,
    XNNPACKQuantizer,
)
from executorch.backends.xnnpack.partition.xnnpack_partitioner import (
    XnnpackPartitioner,
)
from executorch.exir import to_edge_transform_and_lower

PYTORCH_MODEL_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/mobilenet_v2.pth"
EXECUTORCH_QUANTIZED_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/models/mobilenet_v2_quantized_xnnpack.pte"
CALIB_IMAGES_DIR = "Image_Lab_ExecuTorch/calib_images"   

# 1) Load FP32 model.
model = models.mobilenet_v2()
model.load_state_dict(torch.load(PYTORCH_MODEL_PATH, map_location="cpu"))
model.eval()

# Example input only defines shapes for export.
example_inputs = (torch.randn(1, 3, 224, 224),)

# 2) Configure XNNPACK quantizer.
qparams = get_symmetric_quantization_config(is_per_channel=True)
quantizer = XNNPACKQuantizer()
quantizer.set_global(qparams)

def skip_classifier_head(n):
    stack = n.meta.get("nn_module_stack", {})
    for (fqn, _typ) in stack.values():
        if isinstance(fqn, (tuple, list)):
            fqn_str = " ".join(map(str, fqn))
        else:
            fqn_str = str(fqn)

        if "classifier.1" in fqn_str:
            return False
    return True

quantizer.set_filter_function(skip_classifier_head)

# 3) Export float model for PT2E and prepare for quantization.
exported = torch.export.export(model, example_inputs)
training_ep = exported.module()
prepared = prepare_pt2e(training_ep, quantizer)
move_exported_model_to_eval(prepared) 

# 4) Calibration with real images using same preprocessing as inference.
calib_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

calib_dataset = datasets.ImageFolder(CALIB_IMAGES_DIR, transform=calib_transform)

calib_loader = torch.utils.data.DataLoader(
    calib_dataset,
    batch_size=1,
    shuffle=False,     
    num_workers=0      
)

print(f"Calibrating on {len(calib_dataset)} images from {CALIB_IMAGES_DIR}...")

num_calib = min(100, len(calib_dataset))
with torch.inference_mode():
    for i, (calib_img, _) in enumerate(calib_loader):
        if i >= num_calib:
            break
        prepared(calib_img)

# 5) Convert calibrated model to quantized model.
quantized_model = convert_pt2e(prepared)

# 6) Export quantized model and lower to XNNPACK, then to ExecuTorch.
exported_quant = export(quantized_model, example_inputs)

et_program = to_edge_transform_and_lower(
    exported_quant,
    partitioner=[XnnpackPartitioner()],
).to_executorch()

# 7) Save .pte and compute sizes.
with open(EXECUTORCH_QUANTIZED_PATH, "wb") as f:
    et_program.write_to_file(f)

pytorch_size = os.path.getsize(PYTORCH_MODEL_PATH) / (1024 * 1024)
quantized_size = os.path.getsize(EXECUTORCH_QUANTIZED_PATH) / (1024 * 1024)

print("\n" + "="*60)
print("MODEL SIZE COMPARISON")
print("="*60)
print(f"PyTorch (FP32):                  {pytorch_size:6.2f} MB")
print(f"ExecuTorch Quantized (INT8):     {quantized_size:6.2f} MB")
print(f"Size reduction:                  {((pytorch_size - quantized_size) / pytorch_size * 100):5.1f}%")
print(f"Savings:                         {pytorch_size - quantized_size:6.2f} MB")
print("="*60)

IMAGE_PATH = "Image_Lab_ExecuTorch/img_class/mobilenet/images/cat.jpg"

run_executorch_model(EXECUTORCH_QUANTIZED_PATH, IMAGE_PATH)

> Note: Slightly higher or lower probabilities in the INT8 model are normal and do not indicate a problem by themselves. Quantization slightly changes the logits, and softmax can become slightly sharper or flatter. You should, however, expect the predictions to remain in the same order.

## **Integration with PiCamera**

> Note: this section only supports Raspberry Pi with a PiCamera.

We essentially have two different Python worlds: system Python (where the camera stack is wired up) and our virtual environment (where ExecuTorch is installed). To run ExecuTorch on live frames from the Pi camera, we need to bridge those worlds.

Why the camera “only works” in system Python
- Recent Raspberry Pi OS uses Picamera2 on top of libcamera as the recommended interface.
- The Picamera2/libcamera Python bindings are usually installed into the system Python and are not trivially pip‑installable into arbitrary virtual environments or other Python versions.
- Once we create a separate virtual environment for Jupyter, it will not automatically see the Picamera2/libcamera bindings that live under system Python, so imports fail, or the camera device is not accessible from that environment.

> We will use a `two‑process solution`: capture in system, infer in venv. 

For that, we should run a small capture service under system Python that:
- Grabs frames from the Pi camera (Picamera2 / libcamera).
- Sends frames to your ExecuTorch process over a local channel (e.g., ZeroMQ, TCP/UDP socket, shared memory, filesystem (write JPEG/PNG to a temp directory and signal)), or a simple HTTP server.
	
The venv process receives the frame, decodes it, runs the preprocessing pipeline (resize, normalize), then calls ExecuTorch for inference.

A very simple example is provided below as a script that can be created, and then run:

In [ ]:
%%writefile Image_Lab_ExecuTorch/camera_capture.py

import numpy as np
from picamera2 import Picamera2
import time

print(f"NumPy version: {np.__version__}")

# Initialize camera.
picam2 = Picamera2()

config = picam2.create_preview_configuration(main={"size":(640,480)}) 
picam2.configure(config)
picam2.start()

# Wait for camera to warm up.
time.sleep(2)

print("Camera ready!")

# Capture image.
picam2.capture_file("Image_Lab_ExecuTorch/camera_capture.jpg")
print("Image captured: camera_capture.jpg")

# Stop camera.
picam2.stop()
picam2.close()

Add the path to your system Python below, and run the camera_capture script:

In [ ]:
%%bash

~/../../usr/bin/python "$(pwd)/Image_Lab_ExecuTorch/camera_capture.py"

Now run the executorch model on your captured image, from the Jupyter kernel:

In [ ]:
run_executorch_model(EXECUTORCH_QUANTIZED_PATH, "Image_Lab_ExecuTorch/camera_capture.jpg")

## **End of Lab Summary – What Have We Learned?**

In this lab, we recapped the deployment workflow of PyTorch models to ExecuTorch, and the benefits of lowering via the XNNPACK backend for Arm-based CPUs. We examined the delegation of the MobileNetV2 CNN to XNNPACK, observing a much cleaner delegation than the transformer-based models in Lab 1. This resulted in more significant reductions in latency. We then extended this learning by moving from FP32 models, to quantized INT8 - typical of model deployments at the Edge and on resource-constrained devices. This demonstrated a significant reduction in .pte file size.

The lab demonstrates that vision/image applications are suitable and even ideal on ExecuTorch, and example code is provided handling pre-processing such as resizing, normalization, and tensor formatting. As well as demonstrating the model's use for classifying pre-provided images, a basic start-point is provided for capturing images on the PiCamera. You are encouraged to attempt more comprehensive applications using the PiCamera to capture images and ExecuTorch to perform classification or detection. You may be interested in exploring ExecuTorch on mobile. You can deploy to mobile Arm-based CPUs, again using XNNPACK. In the latest Arm-based mobile phones, with SME2 enabled, you can expect very good performance. You are recommended to look at these learning paths, which provide a good foundation to extending what you have learnt to the mobile domain:
- [Build an Android Chat App](https://learn.arm.com/learning-paths/mobile-graphics-and-gaming/build-llama3-chat-android-app-using-executorch-and-xnnpack/)
- [Profile ExecuTorch Models on SME2](https://learn.arm.com/learning-paths/cross-platform/sme-executorch-profiling/)

In the next lab, we will move beyond CPU execution to explore alternative backend hardware. Lab 3 will focus on the Arm Ethos-U Neural Processing Unit (NPU), a purpose-built AI accelerator designed for Edge and embedded use cases. Using MobileNetV2 as our reference model, we will demonstrate how to export and lower a PyTorch model to the Ethos-U backend, apply backend-specific quantization, and compile the model for NPU execution. We will also walk through deployment using a Fixed Virtual Platform (FVP) simulation, enabling us to validate execution in a representative hardware environment without requiring physical silicon. 

In addition, the lab will introduce Google’s Model Explorer tool, along with several adapters developed by Arm to inspect and analyze `.pte` models and models expressed in the TOSA intermediate representation. We will discuss how TOSA functions as a hardware-agnostic intermediate layer in the compilation pipeline, and how it enables portability and structured lowering toward accelerator backends like Ethos-U.

---

This lab was developed by Prof. Marcelo Rovai (UNIFEI University)

**Resources:**
- [EXECUTORCH Documentation](https://pytorch.org/executorch/)
- [EdgeML Made Easy Book](https://mjrovai.github.io/EdgeML_Made_Ease_ebook/)